In [1]:
## IMPORTS
from pystac_client import Client
import planetary_computer as pc
from shapely.geometry import box
import rioxarray
import os

In [2]:
# Coordinates for a small area in Mumbai (Longitude, Latitude)
# Format: (min_lon, min_lat, max_lon, max_lat)
area_of_interest = box(72.8, 19.0, 72.9, 19.1)
time_range = "2023-01-01/2023-01-31"
# Connect to the Planetary Computer Catalog
catalog = Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace
)
print(f"Searching for data over: {area_of_interest.bounds} during {time_range}")

Searching for data over: (72.8, 19.0, 72.9, 19.1) during 2023-01-01/2023-01-31


In [5]:
# Cell 3
search_s2 = catalog.search(
    collections=["sentinel-2-l2a"],
    intersects=area_of_interest,
    datetime=time_range,
    query={"eo:cloud_cover": {"lt": 10}} # Less than 10% clouds!
)

items_s2 = list(search_s2.items())
print(f"Found {len(items_s2)} clear Sentinel-2 images.")

# Grab the clearest one
best_s2_image = items_s2[0]
print(f"Selected Image Date: {best_s2_image.datetime}")


Found 20 clear Sentinel-2 images.
Selected Image Date: 2023-01-27 05:41:09.024000+00:00


In [6]:
# Cell 4
# Create a folder to save our data
os.makedirs("data", exist_ok=True)

# The bands we want to download (RGB + NIR)
bands_to_download = ["B04", "B03", "B02", "B08"]

for band in bands_to_download:
    url = best_s2_image.assets[band].href
    print(f"Downloading {band}...")
    
    # Open remote file, crop to our area, and save
    rds = rioxarray.open_rasterio(url)
    clipped = rds.rio.clip([area_of_interest], crs="EPSG:4326")
    clipped.rio.to_raster(f"data/s2_{band}.tif")
    
print("Optical Data Downloaded!")


Optical Data Downloaded!


In [3]:
# Cell 5
search_s1 = catalog.search(
    collections=["sentinel-1-rtc"], # RTC means radiometrically terrain corrected (flattened)
    intersects=area_of_interest,
    datetime=time_range
)

items_s1 = list(search_s1.items())
best_s1_image = items_s1[0]
print(f"Found Sentinel-1 Image Date: {best_s1_image.datetime}")


Found Sentinel-1 Image Date: 2023-01-30 01:03:38.147142+00:00


In [ ]:
# Cell 6
for band in ["vh", "vv"]:
    url = best_s1_image.assets[band].href
    print(f"Downloading SAR {band}...")
    
    rds = rioxarray.open_rasterio(url)
    clipped = rds.rio.clip([area_of_interest], crs="EPSG:4326")
    clipped.rio.to_raster(f"data/s1_{band}.tif")
    
print("SAR Data Downloaded!")



In [ ]:
# Cell 6 (Updated)
# We removed "vv" from the list since you already have it!
for band in ["vh"]:
    url = best_s1_image.assets[band].href
    print(f"Downloading SAR {band}...")
    
    rds = rioxarray.open_rasterio(url)
    clipped = rds.rio.clip([area_of_interest], crs="EPSG:4326")
    clipped.rio.to_raster(f"data/s1_{band}.tif")
    
print("SAR Data Downloaded!")
